# Module optionnel CoastSat - téléchargement, configuration et extraction du trait de côte

Ce notebook complète la session `06-coastal-monitoring`. Il montre comment préparer une zone d'étude, interroger Google Earth Engine via CoastSat, télécharger des images satellite, extraire un trait de côte test, puis préparer une correction de marée avec FES2022.

Il doit être exécuté dans l'environnement séparé `coastsat`, pas dans `ggosss2026`. Les téléchargements et l'extraction complète sont désactivés par défaut pour éviter de générer des fichiers lourds pendant l'atelier.

## 1. Démarrage recommandé

Depuis un terminal local :

```bash
conda activate coastsat
bash ~/CoastSat_workspace/start_jupyter_coastsat.sh
```

Si Earth Engine n'est pas encore authentifié :

```bash
earthengine authenticate
```

ou, selon la configuration Google :

```bash
gcloud auth application-default login
```

In [ ]:
# Bloc configuration: importer les modules et vérifier les chemins CoastSat locaux.
from pathlib import Path
import os
import sys
import json
import pickle
import warnings

warnings.filterwarnings("ignore")

COASTSAT_SRC = Path(os.environ.get("COASTSAT_SRC", Path.home() / "PythonPackages" / "CoastSat-master"))
COASTSAT_WORKSPACE = Path(os.environ.get("COASTSAT_WORKSPACE", Path.home() / "CoastSat_workspace"))

if COASTSAT_SRC.exists():
    sys.path.insert(0, str(COASTSAT_SRC))

from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_tools

print("COASTSAT_SRC =", COASTSAT_SRC)
print("COASTSAT_WORKSPACE =", COASTSAT_WORKSPACE)
print("ROI folder exists =", (COASTSAT_WORKSPACE / "ROI").exists())
print("Data folder exists =", (COASTSAT_WORKSPACE / "data").exists())

## 2. Choisir une emprise de travail

Pour une formation, on évite de commencer avec une grande zone. On privilégie une petite emprise :

- `benin_cotonou_channel` pour connecter la session au contexte local de Cotonou et du lac Nokoué ;
- `cameroon_kribi_demo` pour réutiliser les résultats déjà disponibles dans le workspace CoastSat ;
- `gulf_of_guinea_box` seulement pour discuter de la logique régionale, pas pour télécharger en bloc.

In [ ]:
# Bloc ROI: définir quelques emprises pédagogiques en longitude/latitude WGS84.
ROI_LIBRARY = {
    "benin_cotonou_channel": {
        "sitename": "BEN_Cotonou_Channel_demo",
        "polygon": [[[2.32, 6.31], [2.48, 6.31], [2.48, 6.43], [2.32, 6.43], [2.32, 6.31]]],
        "comment": "Petite emprise autour du chenal de Cotonou et du débouché lagune-océan.",
    },
    "benin_lake_nokoue": {
        "sitename": "BEN_Lake_Nokoue_demo",
        "polygon": [[[2.25, 6.35], [2.65, 6.35], [2.65, 6.58], [2.25, 6.58], [2.25, 6.35]]],
        "comment": "Emprise large pour discuter lac Nokoué; à réduire avant téléchargement réel.",
    },
    "cameroon_kribi_demo": {
        "sitename": "Kribi_Z1",
        "polygon": [[[9.70, 2.85], [9.86, 2.85], [9.86, 3.05], [9.70, 3.05], [9.70, 2.85]]],
        "comment": "Emprise test proche des sorties CoastSat déjà présentes localement.",
    },
    "gulf_of_guinea_box": {
        "sitename": "Gulf_of_Guinea_context_only",
        "polygon": [[[-8.0, -6.0], [14.5, -6.0], [14.5, 8.0], [-8.0, 8.0], [-8.0, -6.0]]],
        "comment": "Contexte régional uniquement; trop grand pour un téléchargement CoastSat pédagogique.",
    },
}

ROI_NAME = "benin_cotonou_channel"
roi = ROI_LIBRARY[ROI_NAME]
polygon = SDS_tools.smallest_rectangle(roi["polygon"])

print(ROI_NAME)
print(roi["comment"])
print(json.dumps(polygon, indent=2))

## 3. Construire les paramètres CoastSat

Pour limiter la taille des téléchargements pendant un test :

- commencer avec Sentinel-2 seulement (`S2`) ;
- limiter la période à quelques mois ;
- garder une petite emprise ;
- vérifier d'abord le nombre d'images disponibles avant tout téléchargement.

In [ ]:
# Bloc paramètres: préparer les entrées CoastSat pour un inventaire ou un téléchargement limité.
inputs = {
    "polygon": polygon,
    "dates": ["2025-01-01", "2025-06-30"],
    "sat_list": ["S2"],
    "sitename": roi["sitename"],
    "filepath": str(COASTSAT_WORKSPACE / "training_downloads"),
}

print(inputs)

In [ ]:
# Bloc sécurité: garder les actions lourdes désactivées par défaut.
RUN_CHECK_AVAILABLE = False
RUN_DOWNLOAD = False
RUN_EXTRACTION = False

print("RUN_CHECK_AVAILABLE =", RUN_CHECK_AVAILABLE)
print("RUN_DOWNLOAD =", RUN_DOWNLOAD)
print("RUN_EXTRACTION =", RUN_EXTRACTION)

In [ ]:
# Bloc inventaire: interroger CoastSat/GEE pour connaître le nombre d'images disponibles.
# Active RUN_CHECK_AVAILABLE seulement si Earth Engine est authentifié et si Internet est disponible.
if RUN_CHECK_AVAILABLE:
    SDS_download.check_images_available(inputs)
else:
    print("Inventaire non lancé. Mettre RUN_CHECK_AVAILABLE = True pour interroger GEE.")

In [ ]:
# Bloc téléchargement: télécharger les images si l'inventaire est acceptable.
# Cette étape peut créer des fichiers volumineux; garder RUN_DOWNLOAD = False pendant la démonstration.
if RUN_DOWNLOAD:
    metadata = SDS_download.retrieve_images(inputs)
else:
    metadata = None
    print("Téléchargement non lancé. Mettre RUN_DOWNLOAD = True pour récupérer les images.")

## 4. Extraction test du trait de côte

Après téléchargement, CoastSat peut prétraiter les images et extraire les lignes de rivage. Pour une séance courte, on recommande de montrer la configuration et de commenter des sorties déjà calculées, puis de lancer une extraction sur une petite ROI seulement si le temps le permet.

In [ ]:
# Bloc extraction: paramètres minimaux pour une extraction CoastSat.
settings = {
    "cloud_thresh": 0.2,
    "dist_clouds": 300,
    "output_epsg": 32631,  # UTM 31N, adapté au Bénin/Cotonou. Utiliser 32632 ou autre selon le site.
    "check_detection": False,
    "adjust_detection": False,
    "save_figure": True,
    "min_beach_area": 500,
    "min_length_sl": 100,
    "cloud_mask_issue": False,
    "sand_color": "default",
    "pan_off": False,
    "s2cloudless_prob": 40,
    "inputs": inputs,
}

if RUN_EXTRACTION:
    if metadata is None:
        metadata = SDS_download.get_metadata(inputs)
    SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)
    settings["reference_shoreline"] = SDS_preprocess.get_reference_sl(metadata, settings)
    settings["max_dist_ref"] = 100
    output = SDS_shoreline.extract_shorelines(metadata, settings)
    output = SDS_tools.remove_duplicates(output)
    output = SDS_tools.remove_inaccurate_georef(output, 10)
    print("shorelines", len(output.get("shorelines", [])))
else:
    print("Extraction non lancée. Mettre RUN_EXTRACTION = True après téléchargement.")

## 5. Lire une sortie CoastSat déjà calculée

Cette partie permet de faire une démonstration sans Internet. Elle utilise les sorties Kribi déjà présentes dans le workspace local si elles existent.

In [ ]:
# Bloc démonstration locale: charger un output CoastSat déjà calculé dans le workspace.
existing_site = "Kribi_Z1"
output_pkl = COASTSAT_WORKSPACE / "data" / existing_site / f"{existing_site}_output.pkl"

if output_pkl.exists():
    with output_pkl.open("rb") as f:
        output_local = pickle.load(f)
    print("site", existing_site)
    print("shorelines", len(output_local.get("shorelines", [])))
    print("dates", len(output_local.get("dates", [])))
    if output_local.get("dates"):
        print("première date", output_local["dates"][0])
        print("dernière date", output_local["dates"][-1])
else:
    print("Sortie locale introuvable:", output_pkl)

## 6. FES2022: stratégie légère pour la marée

Le modèle FES2022 global est trop lourd pour GitHub et trop volumineux pour une distribution simple. Pour le workshop, on recommande trois niveaux :

1. utiliser des séries de marée CSV déjà préparées pour la démonstration ;
2. garder FES2022 global sur une machine instructeur ou un disque externe ;
3. créer un sous-ensemble régional Bénin/lac Nokoué ou Golfe de Guinée avec `prepare_fes2022_regional_subset.py`.

Exemple :

```bash
conda activate coastsat
python notebooks/prepare_fes2022_regional_subset.py \
  --fes-root /chemin/vers/fes2022b \
  --out-root /chemin/vers/fes2022b_gulf_of_guinea_subset \
  --region gulf_of_guinea
```

Pour un test plus léger autour du Bénin :

```bash
python notebooks/prepare_fes2022_regional_subset.py \
  --fes-root /chemin/vers/fes2022b \
  --out-root /chemin/vers/fes2022b_benin_subset \
  --region benin_nokoue
```

In [ ]:
# Bloc contrôle FES: vérifier que pyfes est disponible sans charger le modèle.
try:
    import pyfes
    print("pyfes disponible")
except Exception as exc:
    print("pyfes indisponible:", repr(exc))

## 7. Résultat attendu pour la formation

À la fin du module, les participants doivent pouvoir expliquer :

- comment choisir une ROI adaptée ;
- pourquoi il faut limiter la période et la taille de la zone au début ;
- comment vérifier le nombre d'images avant téléchargement ;
- comment passer du téléchargement à l'extraction du trait de côte ;
- pourquoi la correction de marée nécessite FES2022 ou une série de marée préparée ;
- pourquoi les gros fichiers restent hors GitHub.